In [2]:
import heapq

# Define the target/goal configuration (0 represents the empty/blank space)
GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

# Precompute target coordinates for faster Manhattan distance lookups
# Format: { tile_number: (row, col) }
GOAL_POSITIONS = {}
for r in range(3):
    for c in range(3):
        GOAL_POSITIONS[GOAL_STATE[r][c]] = (r, c)

class Node:
    def __init__(self, state, parent=None, action=None, g=0):
        self.state = state      # 2D tuple representation of the board
        self.parent = parent    # Link to parent node to reconstruct the path
        self.action = action    # The move made to reach this state ('Up', 'Down', 'Left', 'Right')
        self.g = g              # Depth / path cost from the start node
        self.h = self.calculate_manhattan_distance()  # Manhattan distance heuristic
        self.f = self.g + self.h                      # Total A* evaluation score

    def calculate_manhattan_distance(self):
        """Calculates h(n): sum of absolute vertical and horizontal distances
        for all tiles from their goal positions, excluding the blank (0)."""
        distance = 0
        for r in range(3):
            for c in range(3):
                val = self.state[r][c]
                # Skip the blank tile
                if val != 0:
                    goal_r, goal_c = GOAL_POSITIONS[val]
                    # Calculate |x1 - x2| + |y1 - y2|
                    distance += abs(r - goal_r) + abs(c - goal_c)
        return distance

    # Comparison operator required for the heapq priority queue wrapper
    def __lt__(self, other):
        return self.f < other.f

def find_blank(state):
    """Finds the (row, col) index of the blank space (0)."""
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c

def get_neighbors(node):
    """Generates all valid successor nodes by shifting the blank space."""
    neighbors = []
    r, c = find_blank(node.state)

    # Possible shifts: (row change, col change, move direction name)
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Convert tuple of tuples to mutable list of lists to perform the swap
            new_state = [list(row) for row in node.state]
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            # Convert back to immutable tuple layout for hashing/set operations
            final_state = tuple(tuple(row) for row in new_state)

            neighbors.append(Node(final_state, parent=node, action=action, g=node.g + 1))

    return neighbors

def solve_a_star(start_state):
    """Solves the 8-puzzle problem using the A* Search algorithm with Manhattan Distance."""
    start_node = Node(start_state)

    # Priority queue (Min-Heap) sorted by the lowest f(n) value
    open_list = []
    heapq.heappush(open_list, start_node)

    # Track states that have already been explored
    closed_set = set()

    while open_list:
        # Get the node with the lowest f(n)
        current_node = heapq.heappop(open_list)

        # Check if the goal state is reached
        if current_node.state == GOAL_STATE:
            return reconstruct_path(current_node)

        closed_set.add(current_node.state)

        # Generate and evaluate all valid child moves
        for neighbor in get_neighbors(current_node):
            if neighbor.state in closed_set:
                continue

            # Add to the priority queue if not already visited
            heapq.heappush(open_list, neighbor)

    return None # Return None if no solution is found

def reconstruct_path(node):
    """Backtracks from the goal node down to the parent to compile the final solution steps."""
    path = []
    current = node
    while current:
        path.append(current)
        current = current.parent
    return path[::-1] # Reverse path to display from Start to Goal


# --- Execution Example ---
if __name__ == "__main__":
    # 0 represents the empty blank tile
    initial_board = (
        (1, 2, 3),
        (4, 0, 6),
        (7, 5, 8)
    )

    print("Starting A* Engine Solution Strategy (Manhattan Distance)...")
    solution_steps = solve_a_star(initial_board)

    if solution_steps:
        print(f"Goal reached successfully! Total moves: {len(solution_steps) - 1}\n")
        for step in solution_steps:
            if step.action:
                print(f"Move Blank: {step.action} (g={step.g}, h={step.h}, f={step.f})")
            else:
                print(f"Initial State (g={step.g}, h={step.h}, f={step.f})")

            # Print the actual grid state
            for row in step.state:
                print(" ".join(str(x) if x != 0 else "_" for x in row))
            print("-" * 15)
    else:
        print("This board configuration is unsolvable.")


Starting A* Engine Solution Strategy (Manhattan Distance)...
Goal reached successfully! Total moves: 2

Initial State (g=0, h=2, f=2)
1 2 3
4 _ 6
7 5 8
---------------
Move Blank: Down (g=1, h=1, f=2)
1 2 3
4 5 6
7 _ 8
---------------
Move Blank: Right (g=2, h=0, f=2)
1 2 3
4 5 6
7 8 _
---------------
